# 03. 마피아 게임 실행 (GameEngine 버전)

**목적**: 순수 파이썬 `GameEngine`으로 마피아 한 판을 진행한다. 두 가지 모드를 지원한다.

| 모드 | 구성 | 설명 |
|---|---|---|
| 관전 모드 | AI 8명 | 사람은 구경만 한다. 마피아의 밤 대화/의사/경찰 선택도 보인다 |
| 참여 모드 | 사람 1 + AI 7 | 사람 정체 정보만 보인다. 발언, 투표를 직접 입력한다. `?질문`으로 사회자에게 규칙 문의 |

`GameEngine`의 역할 분담 (**LLM은 연기, 진실은 코드가 지킨다**)

- 엔진(코드): 밤 처리, 사망 발표, 투표 집계, 동점 재투표, 승리 판정, 정보 분리
- 컨트롤러(LLM / 사람): 발언과 선택만 담당
- 게임이 끝나면 `reveal_report()`로 전원의 역할과 속마음을 공개하고, `save_log()`로 JSON 저장

진행: 밤 1 -> 낮 1(토론) -> 투표 -> 밤 2 -> ... (마피아 전멸 시 시민 승, 마피아 수 >= 시민 수 시 마피아 승, 10일 초과 무승부)

In [1]:
# ===== 0. 작업 폴더 맞추기 =====
# 이 노트북은 08_Mafia_Agent 폴더에서 실행한다고 가정한다.
# (VS Code에서 상위 폴더를 열었을 때도 동작하도록 폴더를 찾아 이동한다.)
import os
import sys
from pathlib import Path

if not Path("config.py").exists():
    if Path("08_Mafia_Agent/config.py").exists():
        os.chdir("08_Mafia_Agent")
    else:
        raise FileNotFoundError("08_Mafia_Agent 폴더에서 노트북을 열어주세요.")

sys.path.insert(0, os.getcwd())

import config

print("===== 작업 폴더 =====")
print(config.BASE_DIR)
print("LLM:", config.LLM_MODEL, "/ 임베딩:", config.EMBED_MODEL)

===== 작업 폴더 =====
.
LLM: qwen3:8b / 임베딩: bge-m3


In [2]:
# ===== 1. 준비 =====
import random
import time

import rag
import mafia_core as core
import mafia_agents as agents

vs = rag.load_db()
print("===== 벡터 DB 로드 완료 (Chunk %d개) =====" % vs._collection.count())

===== 벡터 DB 로드 완료 (Chunk 23개) =====


## 1. 관전 모드: AI 8명

`build_controllers()`는 플레이어마다 역할에 맞는 도구와 캐릭터 페르소나를 가진 `LLMPlayer`를 만들어 준다.
**한 판에 수 분이 걸린다.** `SEED`를 바꾸면 다른 역할 배정이 나온다.

In [3]:
# ===== 2. 관전 모드 실행 =====
SEED = 11

state = core.new_game(list(config.CHARACTERS), None, seed=SEED)
controllers = agents.build_controllers(state, vs)
engine = core.GameEngine(state, controllers)   # on_event 기본값: 한국어로 출력

t0 = time.time()
winner = engine.run()
elapsed = time.time() - t0

print("\n===== 게임 종료 =====")
print("승자:", winner, "| 진행 일수:", state.day, "| 소요 시간: %.0f초" % elapsed)


===== 마피아 게임 시작 (8명) =====
참가자: 강태호, 윤서진, 박준수, 최민지, 정도현, 한소율, 오지환, 임채원
  [비밀 · 마피아 강태호] 당신의 직업: 마피아
  [비밀 · 마피아 강태호] 동료 마피아: 윤서진
  [비밀 · 마피아 윤서진] 당신의 직업: 마피아
  [비밀 · 마피아 윤서진] 동료 마피아: 강태호
  [비밀 · 경찰 박준수] 당신의 직업: 경찰
  [비밀 · 시민 최민지] 당신의 직업: 시민
  [비밀 · 의사 정도현] 당신의 직업: 의사
  [비밀 · 시민 한소율] 당신의 직업: 시민
  [비밀 · 시민 오지환] 당신의 직업: 시민
  [비밀 · 시민 임채원] 당신의 직업: 시민

===== 1일차 밤 =====
  [비밀 · 마피아 강태호] 박준수를 노립니다 (이유: 박준수는 경찰/의사로 보이는 사람으로, 시민팀에 위협이 되는 사람입니다.) [제안]
  [비밀 · 마피아 윤서진] 박준수를 노립니다 (이유: 경찰/의사로 보이는 사람으로 시민팀에 위협이 되는 사람입니다.) [최종]
  [비밀 · 의사 정도현] 정도현 보호
박준수(경찰)이 밤에 사망
  [비밀 · 경찰 박준수] 강태호 조사 결과: 마피아다

===== 1일차 낮 토론 =====
[최민지] 강태호, 왜 밤에 박준수를 처형했어요? 이유가 뭐예요?
[오지환] 강태호야! 왜 박준수를 처형했지? 눈빛이 이상해! 느낌이 와!
[강태호] 야, 최민지야. 너 아까 박준수 처형 이유를 물어봤잖아. 왜 나한테만 물어봤어? 네가 의심하는 사람이 누구야?
[임채원] 앞선 날에는 강태호가 처형을 결정한 사람일 가능성이 큽니다. 강태호가 다른 사람에게 이유를 묻는 행동이 의심스럽습니다.
[정도현] 강태호, 밤에 처형 결정한 사람인데 낮에 말 많아. 왜?
[한소율] 윤서진 씨, 밤에 무슨 일이 있었는지 조금만 말씀해 주실 수 있을까요? 다들 조금만 차분하게 이야기해 봐요.
[윤서진] 강태호 씨, 밤에 처형 결정을 내리고 낮에 많은 말을 하며 다른 사람을 의심하는 태도는 시민이기 어려운

## 2. 게임 종료 후: 속마음 공개, 로그 저장, 통계

In [4]:
# ===== 3. 결과 공개 =====
print(core.reveal_report(state))

===== 게임 결과: 시민팀 승리 (3일차) =====

[역할 공개]
  강태호 : 마피아 (사망)
  윤서진 : 마피아 (사망)
  박준수 : 경찰 (사망)
  최민지 : 시민 (사망)
  정도현 : 의사 (생존)
  한소율 : 시민 (생존)
  오지환 : 시민 (사망)
  임채원 : 시민 (사망)

[밤 기록]
  1일차: 처치 박준수, 보호 정도현, 사망 박준수, 경찰 조사 강태호(마피아)
  2일차: 처치 최민지, 보호 윤서진, 사망 최민지
  3일차: 처치 오지환, 보호 윤서진, 사망 오지환

[속마음 요약]
  <강태호 / 마피아>
    1일차 night: [밤 처치 선택] 박준수: 박준수는 경찰/의사로 보이는 사람으로, 시민팀에 위협이 되는 사람입니다.
    1일차 day: 야, 박준수를 처형했으면 뭔가 이유가 있어야 하는 거지? 최민지랑 오지환도 의심하는 눈빛으로 보는 거야. 윤서진은 믿을 만해. 나도 그쪽으로 의... 의심: 최민지 40, 오지환 30
    1일차 vote: [투표] 최민지: 최민지가 밤 사이에 박준수를 처형한 것에 대해 의심스럽고, 발언에서도 의심하는 눈빛을 보였기 때문입니다.
  <윤서진 / 마피아>
    2일차 vote: [투표] 한소율: 한소율은 밤에 많은 행동을 했고, 강태호의 처형 결정에 대해 의심스러운 태도를 보였습니다. 이는 시민이기 어려운 행동입니다.
    2일차 vote: [재투표] 임채원: 임채원은 투표에서 윤서진을 지목하며, 윤서진을 의심하는 태도를 보였습니다. 이는 시민이기 어려운 행동입니다.
    3일차 night: [밤 처치 선택] 오지환: 오지환은 1일차와 2일차에 강태호를 의심하는 태도를 보였고, 행동이 불일치한 점이 있었습니다. 또한, 2일차 투표에서...
    3일차 day: 현재까지의 투표 패턴을 살펴보면, 윤서진은 1일차에 강태호를 지목했고, 2일차에는 아무 말 없이 한소율을 투표했습니다. 이는 행동 불일치로 보입... 의심: 한소율 70, 정도현 60
    3일차 vote

In [5]:
# ===== 4. 로그 저장 + 통계 =====
log_path = core.save_log(state)
print("===== 로그 저장 완료 =====")
print("저장 위치:", log_path)

print("\n===== 안전장치 통계 =====")
print("AI 결정 요청 수 :", state.stats["llm_calls"])   # 발언/투표/밤 행동 요청 수 (실제 LLM 호출은 더 많음)
print("재시도 횟수      :", state.stats["retries"])
print("fallback 횟수    :", state.stats["fallbacks"])
print("정체 노출 차단   :", state.stats["leak_blocked"])
print("일차당 평균 소요 : %.0f초" % (elapsed / max(state.day, 1)))

===== 로그 저장 완료 =====
저장 위치: .\logs\mafia_20261118_103556_seed11.json

===== 안전장치 통계 =====
AI 결정 요청 수 : 43
재시도 횟수      : 0
fallback 횟수    : 0
정체 노출 차단   : 0
일차당 평균 소요 : 59초


## 3. 참여 모드: 사람 1 + AI 7

- 사람의 이름은 `config.HUMAN_NAME`(기본 "서지우", 본인 이름으로 바꿔도 됨), AI는 캐릭터 8명 중 7명이 무작위로 참가한다.
- 입력 방법: **번호 또는 이름**을 입력한다. 발언은 자유롭게 문장을 입력한다 (빈 입력은 패스).
- `?질문`: 사회자(룰북 RAG, `type=rule` 필터)에게 규칙을 물어본다. 예: `?경찰은 누구를 조사할 수 있나요`
- 사람이 처치/보호/조사 대상을 정하는 밤에는 본인의 비공개 정보만 화면에 나온다.

> 실행하면 입력창이 뜬다. 입력을 하지 않고 넘어가려면 이 셀은 건너뛰어도 된다.

In [6]:
# ===== 5. 참여 모드 실행 =====
HUMAN_SEED = None   # 숫자를 넣으면 같은 배정으로 재현된다

ai_names = random.Random(HUMAN_SEED).sample(config.CHARACTERS, 7)
h_state = core.new_game(ai_names, config.HUMAN_NAME, seed=HUMAN_SEED)
h_controllers = agents.build_controllers(h_state, vs)   # 사람은 HumanController (input 사용)
h_engine = core.GameEngine(h_state, h_controllers)

h_winner = h_engine.run()
print("\n승자:", h_winner)


===== 마피아 게임 시작 (8명) =====
참가자: 정도현, 박준수, 강태호, 한소율, 윤서진, 오지환, 최민지, 서지우
  [비밀 · 경찰 서지우] 당신의 직업: 경찰

===== 1일차 밤 =====
----------------------------------------
[1일차 night] 당신은 경찰 입니다.
생존자: 정도현, 박준수, 강태호, 한소율, 윤서진, 오지환, 최민지, 서지우
조사할 대상을 고르세요.
  1. 정도현
  2. 박준수
  3. 강태호
  4. 한소율
  5. 윤서진
  6. 오지환
  7. 최민지
번호 또는 이름 (?질문 으로 규칙 문의) > ?의사는 자기 자신을 보호할 수 있나요?   ← (자동 응답)
[사회자] 네, 의사는 자기 자신을 보호할 수 있으며, 연속으로 보호할 수도 있습니다.
(룰북 p.3) (룰북 p.4)
번호 또는 이름 (?질문 으로 규칙 문의) > 4   ← (자동 응답)
밤사이 아무도 사망하지 않았습니다
  [비밀 · 경찰 서지우] 한소율 조사 결과: 마피아다

===== 1일차 낮 토론 =====
[윤서진] 현재까지의 발언은 없으므로 논리적으로 추리하겠습니다. 의심할 대상은 아직 없습니다. 근거가 부족합니다. 첫 낮은 정보가 적으니 특이한 사람을 찾겠습니다.
[오지환] 윤서진아! 왜 아무도 의심 안 했는데 그런 말을 했어? 느낌이 와! 진짜로!
[한소율] 오지환 씨, 혹시 윤서진 씨가 아무도 의심하지 않았다고 말했을 때, 왜 그런 반응을 보이셨나요? 조금 의심돼요. 윤서진 씨는 시민일 수도 있을 텐데, 오지환 씨는 왜 그런 말을 했는지 궁금해요. 다
----------------------------------------
[1일차 day] 당신은 경찰 입니다.
생존자: 정도현, 박준수, 강태호, 한소율, 윤서진, 오지환, 최민지, 서지우
[비공개] 1일차 밤 조사 결과: 한소율은 마피아입니다
발언을 입력하세요 (?질문 으로 규칙 문의, 빈 입력은 패스) > 저는 시민입니다. 오지환님은 투표할 때 

In [7]:
# ===== 6. 참여 모드 결과 공개 =====
print(core.reveal_report(h_state))
print("로그:", core.save_log(h_state))

===== 게임 결과: 마피아팀 승리 (6일차) =====

[역할 공개]
  정도현 : 시민 (생존)
  박준수 : 시민 (사망)
  강태호 : 시민 (사망)
  한소율 : 마피아 (생존)
  윤서진 : 마피아 (사망)
  오지환 : 시민 (사망)
  최민지 : 의사 (사망)
  서지우 : 경찰 (사망) [사람]

[밤 기록]
  1일차: 처치 최민지, 보호 최민지, 의사가 살림, 경찰 조사 한소율(마피아)
  2일차: 처치 서지우, 보호 윤서진, 사망 서지우, 경찰 조사 강태호(비마피아)
  3일차: 처치 최민지, 보호 최민지, 의사가 살림
  4일차: 처치 강태호, 보호 강태호, 의사가 살림
  5일차: 처치 최민지, 보호 최민지, 의사가 살림
  6일차: 처치 최민지, 보호 정도현, 사망 최민지

[속마음 요약]
  <정도현 / 시민>
    4일차 day: 강태호가 오지환을 처형했고, 윤서진을 지목한 행동이 마피아 같아. 윤서진이 처형되면 강태호가 마피아일 가능성 높아. 한소율은 강태호를 의심했고,... 의심: 강태호 80, 한소율 60
    4일차 vote: [투표] 강태호: 강태호는 오지환 처형 시 투표에 참여했고, 윤서진 처형 시에도 지목하며 행동이 불일치하고 마피아 징후가 있어
    5일차 day: 오지환 처형 때 강태호, 박준수, 정도현, 한소율이 지목했고, 윤서진 처형 때도 강태호, 박준수, 한소율이 지목했어. 강태호가 오지환을 처형하고... 의심: 한소율 70, 최민지 60
    5일차 vote: [투표] 한소율: 한소율은 오지환 처형 시 투표에 참여했고, 윤서진 처형 시에도 지목하며 행동이 불일치하고 마피아 징후가 있어
    5일차 vote: [재투표] 한소율: 한소율은 오지환 처형 시 투표에 참여했고, 윤서진 처형 시에도 지목하며 행동이 불일치하고 마피아 징후가 있어
  <박준수 / 시민>
    1일차 vote: [투표] 오지환: 오지환은 처음부터 아무도 의심 안 했다고 말했지만, 그 말이 오히려 마피아의 행동을 숨기기 위한 

In [8]:
print("===== 03 완료: 이제 04_langgraph_mafia.ipynb 로 이동하세요 =====")

===== 03 완료: 이제 04_langgraph_mafia.ipynb 로 이동하세요 =====
